# Medical NER — pipeline MẠNH (DAPT + LLM-gen + fine-tune). Kaggle GPU.

**TRƯỚC `Run All`:** Settings → **GPU T4 x2** (hoặc P100) + **Internet ON**.
Thời gian ~5–6h (trong quota). Cuối cùng tải **`output_ner.zip`** để nộp.

Luồng: DAPT (khép domain gap) → sinh data (template + LLM) → fine-tune encoder → inference.


In [ ]:
# 1) Clone repo
import os, subprocess
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
# 2) Deps (torch+transformers có sẵn). bitsandbytes cho LLM 4-bit.
!pip install -q rapidfuzz pyyaml regex accelerate bitsandbytes
import rapidfuzz, yaml, regex, torch, transformers
print("rapidfuzz",rapidfuzz.__version__,"| torch",torch.__version__,"| CUDA",torch.cuda.is_available())
print("transformers",transformers.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# 3) Test nhanh (đảm bảo code lành)
!python -m pytest -q 2>&1 | tail -3


In [ ]:
# 4) DAPT — MLM tiếp trên 100 file test THẬT (không nhãn) -> khép domain gap. ~10'
!python scripts/dapt.py --model xlm-roberta-large --epochs 8 --batch_size 8 --out models/dapt


In [ ]:
# 5) Sinh data TEMPLATE (nhanh, chắc chắn có)
!python scripts/gen_synth.py --n_train 8000 --n_dev 600


In [ ]:
# 6) (TÙY CHỌN) Sinh data LLM. MẶC ĐỊNH TẮT — HF backend QUÁ CHẬM (~11h/3k, không kịp).
#    DAPT + template đã là pipeline mạnh; KHÔNG bật vẫn tốt.
#    Muốn bật: cài vLLM (nhanh ~20x, ~40'/6k) rồi bỏ comment 3 dòng dưới:
# !pip install -q vllm
# !python scripts/gen_llm.py --backend vllm --n 6000 --out data/synthetic/llm.jsonl \
#    || echo "LLM-gen skip -> train bằng template"


In [ ]:
# 7) XÂY KHO MÃ (RxNav thuốc + ICD)
!python scripts/build_kb.py --rxnorm --icd


In [ ]:
# 8) FINE-TUNE NER — khởi từ DAPT + gộp data (template + LLM). ~2.5h.
!python scripts/train_ner.py --model models/dapt \
   --train "data/synthetic/train.jsonl,data/synthetic/llm.jsonl" \
   --epochs 3 --batch_size 8 --grad_accum 2 --max_length 384 --optim adamw_torch \
   --out models/ner


In [ ]:
# 9) Inference: NER + baseline -> zip
!python scripts/predict.py --pipeline ner --model_dir models/ner --out_dir output_ner --zip
!python scripts/predict.py --pipeline baseline --out_dir output --zip


In [ ]:
# 10) Đo trên dev synthetic (tham khảo — KHÔNG phải điểm thật)
!python scripts/evaluate.py --gold data/synthetic/dev.jsonl --pipeline "ner:models/ner"


In [ ]:
# 11) So file văn xuôi 8.txt: baseline vs NER
import json
print("BASELINE 8:", json.dumps(json.load(open("output/8.json")),ensure_ascii=False)[:300])
print("NER      8:", json.dumps(json.load(open("output_ner/8.json")),ensure_ascii=False)[:300])


In [ ]:
# 12) Copy zip ra /kaggle/working để tải
import shutil, os
for z in ["output_ner.zip","output.zip"]:
    if os.path.exists(z): shutil.copy(z, f'/kaggle/working/{z}')
print("Tải output_ner.zip ở panel Output để nộp.")


## Ghi chú TRUNG THỰC
- **DAPT** = đòn chính khép domain gap (NER synthetic từng sập 0.91→0.19 trên thật). Chạy MLM trên chính 100 file test.
- **LLM (Qwen2.5-7B)** = ANNOTATOR sinh data đa dạng, KHÔNG phải extractor (research: encoder fine-tuned > LLM few-shot ở NER span). Nhãn grounded qua parse_marked.
- **Trích xuất** = encoder XLM-R large fine-tuned + sliding-window (kiểu người thắng n2c2).
- **candidates** = RxNav(thuốc)+ICD(bệnh); cắm RxNorm RRF + ICD-VN đầy đủ để tăng mạnh.
- Điểm `evaluate` là dev synthetic (lạc quan). Điểm THẬT chỉ có trên leaderboard.
- Nếu OOM khi fine-tune large: giảm `--batch_size 4`. Nếu LLM-gen lỗi: pipeline vẫn chạy bằng data template.
